# Centralised XGBoost & Random Forest: NF-ToN-IoT-FHF vs NF-ToN-IoT v1, 1500 rows per label

Same 1500-per-label sets as the FedGATSage run (`fedgatsage-nf-vs-fhf`). The `centralised/` scripts of `asfi50/Fed_GNN` (branch `centralised`) run unchanged, with `--samples-per-class 1500 --min-raw-count 1` so that all labels are kept, as in the FedGATSage run. Split: 20 % test by flow vector.


In [ ]:
REPO_URL = "https://github.com/asfi50/Fed_GNN"
BRANCH = "centralised"

!rm -rf /kaggle/working/Fed_GNN
!git clone --branch {BRANCH} --single-branch {REPO_URL} /kaggle/working/Fed_GNN
%cd /kaggle/working/Fed_GNN/centralised

In [ ]:
!pip install -q -r requirements.txt

## 1. Datasets

NF-ToN-IoT-FHF (`mtasfi/nftoniot-fhf`: NF-ToN-IoT v1 with `Attack`/`Label` taken from matched ToN-IoT PCAP/Zeek flows) and, for comparison with identical settings, the original NF-ToN-IoT v1 (`mtasfi/nftoniot`). Downloaded via signed links.


In [ ]:
import os, urllib.request
os.makedirs("/kaggle/working/raw", exist_ok=True)
SRC = {"fhf": "<signed link from Kaggle MCP download_dataset>",
       "nfv1": "<signed link from Kaggle MCP download_dataset>"}
RAW = {}
for name, url in SRC.items():
    RAW[name] = f"/kaggle/working/raw/{name}.csv"
    urllib.request.urlretrieve(url, RAW[name])
    print(name, RAW[name], f"{os.path.getsize(RAW[name]) / 1e6:.1f} MB")

# Same 1500-per-label sets as the FedGATSage run (fedgatsage-nf-vs-fhf): identical sampling code and seed
import pandas as pd
PER_CLASS, SEED = 1500, 42
for name in list(RAW):
    df = pd.read_csv(RAW[name])
    bal = (df.groupby("Attack", group_keys=False)
             .apply(lambda g: g.sample(n=min(PER_CLASS, len(g)), random_state=SEED))
             .sample(frac=1, random_state=SEED).reset_index(drop=True))
    RAW[name] = f"/kaggle/working/raw/{name}_bal1500.csv"
    bal.to_csv(RAW[name], index=False)
    print(name, len(bal), bal.Attack.value_counts().to_dict())

## 2. Comet ML

Uses the same API key/project convention as `experiments/fedgatsage_experiment.py` in the repo (`centralised/comet_utils.py` reads `COMET_API_KEY` from the environment, falling back to that default). Override it here if you want to log to your own workspace.

In [ ]:
import os
# os.environ["COMET_API_KEY"] = "your-key-here"  # optional override

In [ ]:
for name, path in RAW.items():
    print(f"===== preprocess {name} =====")
    !python preprocess.py --input "{path}" --output data/{name}_balanced.csv --samples-per-class 1500 --min-raw-count 1

In [ ]:
import pandas as pd
for name in RAW:
    print(name); print(pd.read_csv(f"data/{name}_balanced.csv")["Attack"].value_counts().to_string(), "\n")

## 4. Train XGBoost

Set `--device cuda` if the notebook has a GPU accelerator attached; otherwise leave `cpu`.

In [ ]:
for name in RAW:
    print(f"===== XGBoost {name} =====")
    !python train_xgboost.py --data data/{name}_balanced.csv --out-dir results_{name} --device cpu

## 5. Train Random Forest

In [ ]:
for name in RAW:
    print(f"===== Random Forest {name} =====")
    !python train_random_forest.py --data data/{name}_balanced.csv --out-dir results_{name}

## 6. FHF vs NF-ToN-IoT v1, same settings


In [ ]:
import json, pandas as pd
rows = []
for name in RAW:
    for model, f in [("xgboost", "xgboost_classification_report.json"), ("random_forest", "random_forest_classification_report.json")]:
        r = json.load(open(f"results_{name}/{f}"))
        rows.append({"data": name, "model": model, "accuracy": r.get("accuracy"),
                     "macro_f1": r["macro avg"]["f1-score"],
                     **{f"f1_{k}": v["f1-score"] for k, v in r.items() if isinstance(v, dict) and k not in ("macro avg", "weighted avg")}})
cmp = pd.DataFrame(rows)
os.makedirs("/kaggle/working/summary", exist_ok=True)
cmp.to_csv("/kaggle/working/summary/centralised_fhf_vs_nfv1_bal1500.csv", index=False)
print(cmp.set_index(["data", "model"]).T.round(3).to_string())